# NB06 — Build03 Current Law Checker · EUR-Lex Tool → Q8 Official Test

**Scientific AI Builder · Regulatory Agent · 2026-10-19**

## Objective
Verify whether **official-source current-law checking** improves Q8 relative to **frozen Build02 static RAG**, while retaining the exact NB05 v2 retrieval implementation and the NB04b common benchmark context.

- **Build02 stays frozen**: 5 local legislation files, `chunks_legal_aware.json`, NB03 BM25/Dense/RRF/diversification, benchmark Q1–Q10, NB04b answers.
- **Build03 intervention**: add a second function tool, `check_current_law(regulation)`, which fetches **only official EUR-Lex pages** at run time and produces evidence records. Existing `search_regulation()` is unchanged.
- **No implied real-time legal certainty**: detecting an updated consolidated version does *not* prove current applicability, article-level content changes or effective dates. When retrieval/metadata fails → `unverified`, not `unchanged`.
- **Q8 is the planned structural test**: “Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?” (fixed wording; no gold checklist in agent prompt).
- No unrestricted browsing, no web search engine dependency, no scraping of arbitrary sites, no fake test results.

**Run order:** dependencies → asset/config audit → exact NB03 retriever → **50/50 stop gate** → offline parser unit tests → optional live official-source probe → register both tools → Q8 paid run → save & compare → manual frozen checklist evaluation.

**Practical note:** EUR-Lex may respond with a JS/blocked page. Such a response is explicitly `unverified`; do not bypass restrictions or infer a date. This notebook has not performed any live EUR-Lex/API execution for you.

## 0. Dependencies and execution rules

Install packages only if missing. Avoid blanket upgrades in Anaconda. Restart the kernel after any install.

`OPENAI_API_KEY` is **not necessary** for the parity audit or EUR-Lex parser/local checks; only the final agent run asks for it.

In [ ]:
# Resolve paths from the repository root, including nested Jupyter kernel CWDs.
import os as _repo_os
from pathlib import Path as _RepoPath
_repo_start = _RepoPath(_repo_os.environ.get("REGULATORY_AGENT_ROOT", _RepoPath.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_repo_start, *_repo_start.parents)
                  if (p / "data/benchmark/benchmark_questions_v1.json").is_file()
                  and (p / "notebooks/build04/NB09_v3.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REGULATORY_AGENT_ROOT to the Regulatory_agent checkout.")
_repo_os.chdir(REPO_ROOT)

# Only when packages are missing:
# %pip install openai-agents sentence-transformers rank-bm25 numpy pandas requests beautifulsoup4

In [1]:
import os, re, json, hashlib, platform
from datetime import datetime, timezone, date
from pathlib import Path
from getpass import getpass
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from agents import Agent, Runner, ModelSettings, function_tool

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "Run from Regulatory_agent repository root (data/ missing)."
MODEL_NAME = "gpt-6-luna"
REASONING_EFFORT = "medium"
print("Python:", platform.python_version(), "| Model:", MODEL_NAME)
print("API key available:", bool(os.getenv("OPENAI_API_KEY")))
print("API key is not required until the Q8 official run.")

c:\Users\SAMSUNG\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Python: 3.13.5 | Model: gpt-6-luna
API key available: False
API key is not required until the Q8 official run.


## 1. Load and fingerprint the **frozen** Build02 / NB04b assets

All compared outputs are loaded rather than regenerated. The Q8 checklist is **evaluation-only**, never given to the agent. Separate run artifacts use `build03_nb06_` prefixes.

In [2]:
FILES = {
    "chunks": __import__("pathlib").Path("data/processed/chunks_legal_aware.json"),
    "benchmark": __import__("pathlib").Path("data/benchmark/benchmark_questions_v1.json"),
    "retrieval_config": __import__("pathlib").Path("data/processed/retrieval_config_nb03_final_hybrid_static_rag.json"),
    "nb03_results": __import__("pathlib").Path("data/processed/retrieval_results_nb03_final_hybrid_static_rag.csv"),
    "nb04b_config": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_experiment_config.json"),
    "nb04b_answers": __import__("pathlib").Path("results/build01_02/nb04b_context_repaired_openai_luna_answers_comparison.csv"),
}
for key, path in FILES.items():
    assert path.is_file(), f"Missing {key}: {path}"

legal_chunks = json.loads(FILES["chunks"].read_text(encoding="utf-8"))
benchmark = json.loads(FILES["benchmark"].read_text(encoding="utf-8"))
retrieval_config = json.loads(FILES["retrieval_config"].read_text(encoding="utf-8"))
nb04b_config = json.loads(FILES["nb04b_config"].read_text(encoding="utf-8"))
nb03_df = pd.read_csv(FILES["nb03_results"], encoding="utf-8-sig")
comparison_df = pd.read_csv(FILES["nb04b_answers"], encoding="utf-8-sig")
questions = benchmark["questions"]
Q8 = questions["Q8"]
benchmark_hash = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert len(legal_chunks) == 1383, "Library changed: investigate before running controlled test"
assert list(questions) == [f"Q{i}" for i in range(1,11)]
assert benchmark_hash == retrieval_config["benchmark_sha256"]
assert nb04b_config["model"] == MODEL_NAME
assert nb04b_config["reasoning_effort"] == REASONING_EFFORT
assert retrieval_config["retrieval_method"] == "hybrid_dense_bm25_rrf_diverse_v1"
assert retrieval_config["top_k"] == 5
assert retrieval_config["rrf_k"] == 60
assert retrieval_config["candidate_depth_per_retriever"] == 50
assert retrieval_config["diversification"]["max_per_group"] == 2
assert set(comparison_df.qid) == set(questions)

FROZEN_Q8_ANSWER = comparison_df.loc[comparison_df.qid=="Q8", "rag_answer"].iloc[0]
print("Benchmark:", benchmark["benchmark_id"])
print("Chunks:", len(legal_chunks), "| Q8:", Q8)
print("Frozen NB04b Q8 answer loaded, characters:", len(FROZEN_Q8_ANSWER))

Benchmark: regulatory_agent_v1_notion_2026-10-08
Chunks: 1383 | Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?
Frozen NB04b Q8 answer loaded, characters: 2752


## 2. **Exact NB03 retriever implementation** (carry forward NB05 v2)

This implementation mirrors original NB03:
- searchable text = title/file/location/article/heading/text
- **BM25 tokenizer** = `[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?`
- normalized embeddings + dot-product, BM25Okapi
- Dense 50 + BM25 50, RRF k=60, source-group cap 2
- rank-ordered Top-5 for frozen test queries.

**No new BM25 or retrieval enhancement is introduced in NB06.**

In [3]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [safe(chunk.get(k)) for k in (
        "regulation_title", "file", "location", "article", "heading", "text"
    )]
    return " | ".join(p for p in parts if p).strip()

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")
def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

legal_texts = [searchable_text(ch) for ch in legal_chunks]
assert all(legal_texts)
EMBEDDING_MODEL = retrieval_config["embedding_model"]
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
legal_embeddings = embedding_model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
bm25 = BM25Okapi([tokenize(t) for t in legal_texts])
print("NB03-equivalent retriever components ready:", EMBEDDING_MODEL)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

NB03-equivalent retriever components ready: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
RRF_K = int(retrieval_config["rrf_k"])
CANDIDATE_DEPTH = int(retrieval_config["candidate_depth_per_retriever"])
MAX_PER_SOURCE_GROUP = int(retrieval_config["diversification"]["max_per_group"])

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))
    return (file_name, article if article else location)

def hybrid_search(query: str, top_k: int = 5):
    query = str(query).strip()
    if not query:
        raise ValueError("query must be non-empty")
    if not (1 <= int(top_k) <= 10):
        raise ValueError("top_k must be in 1..10")
    q_emb = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    dense_scores = legal_embeddings @ q_emb
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=float)
    dense_order = np.argsort(dense_scores)[::-1][:CANDIDATE_DEPTH]
    bm25_order = np.argsort(bm25_scores)[::-1][:CANDIDATE_DEPTH]
    fused_scores = {}
    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = fused_scores.get(idx, 0.) + 1. / (RRF_K + rank)
    fused_order = sorted(fused_scores, key=fused_scores.get, reverse=True)
    selected, counts = [], {}
    for idx in fused_order:
        chunk = legal_chunks[idx]
        group = source_group(chunk)
        if counts.get(group, 0) >= MAX_PER_SOURCE_GROUP:
            continue
        selected.append((idx, chunk, fused_scores[idx]))
        counts[group] = counts.get(group, 0) + 1
        if len(selected) == int(top_k):
            break
    results = []
    for rank, (idx, chunk, rrf_score) in enumerate(selected, start=1):
        results.append({
            "rank": rank, "chunk_index": int(idx),
            "chunk_id": chunk.get("chunk_id"), "rrf_score": float(rrf_score),
            "file": chunk.get("file"), "regulation_title": chunk.get("regulation_title"),
            "article": chunk.get("article"), "location": chunk.get("location"),
            "heading": chunk.get("heading"), "text": chunk.get("text"),
        })
    return results

## 3. STOP-GATE · rank-ordered NB03 ↔ NB06 parity: **50/50**

Cannot continue to controlled Q8 if any chunk ID/order differs. Save a separate audit file; frozen NB03 outputs are read-only.

In [5]:
PARITY_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb06_retrieval_parity_audit.csv")
audit_rows, parity_by_qid = [], {}
for qid, question in questions.items():
    baseline = nb03_df[nb03_df.qid == qid].sort_values("rank")
    expected = baseline.chunk_id.astype(str).tolist()
    actual = [r["chunk_id"] for r in hybrid_search(question, top_k=5)]
    assert len(expected) == len(actual) == 5
    parity_by_qid[qid] = expected == actual
    for rank, (a,b) in enumerate(zip(expected,actual), start=1):
        audit_rows.append({"qid":qid,"rank":rank,"nb03_chunk_id":a,"nb06_chunk_id":b,"match":a==b})
    print(f"{qid}: {'PASS' if parity_by_qid[qid] else 'MISMATCH'}")
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(PARITY_PATH, index=False, encoding="utf-8-sig")
matched = int(audit_df.match.sum())
PARITY_OK = matched == 50 and all(parity_by_qid.values())
print(f"\nPARITY: {matched}/50 ordered chunks | Saved: {PARITY_PATH}")
assert PARITY_OK, "STOP: original NB03 retrieval parity failed. No official Q8 run allowed."

Q1: PASS
Q2: PASS
Q3: PASS
Q4: PASS
Q5: PASS
Q6: PASS
Q7: PASS
Q8: PASS
Q9: PASS
Q10: PASS

PARITY: 50/50 ordered chunks | Saved: data\build03_nb06_retrieval_parity_audit.csv


## 4. Current-law source registry — **five frozen EU regulations only**

Stable source registry maps known basic acts to their EUR-Lex **basic-act ELI / CELEX** page. We look for publicly surfaced *consolidation metadata* rather than attempting to infer an article-level change from search snippets.

Each recorded snapshot date is the date encoded by the existing local HTML filename, not proof of when an amendment was legally effective. `2019/1013` is a **Commission Implementing Regulation**.

**Security/quality guards:** no caller-provided URL, only allowlisted regulation IDs, only HTTPS/known EUR-Lex host, request timeout and bounded HTML size, inspect final URL, reject invalid 200 responses. If the site blocks access, mark `unverified` rather than fabricating metadata.

In [6]:
LAW_REGISTRY = {
    "2023/915": {"celex":"32023R0915", "eli":"reg/2023/915", "snapshot_date":"2026-08-19", "snapshot_file":"EU_2023_915_consolidated_2026-08-19.html", "title":"Contaminants maximum levels"},
    "2019/1793": {"celex":"32019R1793", "eli":"reg_impl/2019/1793", "snapshot_date":"2026-06-30", "snapshot_file":"EU_2019_1793_consolidated_2026-06-30.html", "title":"Increased import controls / special conditions"},
    "2017/625": {"celex":"32017R0625", "eli":"reg/2017/625", "snapshot_date":"2025-01-05", "snapshot_file":"EU_2017_625_consolidated_2025-01-05.html", "title":"Official controls"},
    "178/2002": {"celex":"32002R0178", "eli":"reg/2002/178", "snapshot_date":"2026-01-01", "snapshot_file":"EU_2002_178_consolidated_2026-01-01.html", "title":"General Food Law"},
    "2019/1013": {"celex":"32019R1013", "eli":"reg_impl/2019/1013", "snapshot_date":None, "snapshot_file":"EU_2019__1013_original.html", "title":"Prior-notification arrangements"},
}

for key, item in LAW_REGISTRY.items():
    assert (__import__("pathlib").Path("data/legal_sources") / item["snapshot_file"]).is_file(), f"Snapshot missing for {key}"
    item["official_basic_url"] = f'https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A{item["celex"]}'

assert set(LAW_REGISTRY) == {"2023/915","2019/1793","2017/625","178/2002","2019/1013"}
print("Current-law checker scope:", list(LAW_REGISTRY))

Current-law checker scope: ['2023/915', '2019/1793', '2017/625', '178/2002', '2019/1013']


## 5. Parser and network client: conservative source verification

The parser accepts only metadata **explicitly present on an accessible official EUR-Lex basic-act page**: consolidated CELEX links and/or an “Access current version (DD/MM/YYYY)” label. A date extracted from a general page body is **not** treated as a definitive current version.

States:
- `metadata_found`: official page accessible and version metadata identified; compare dates only.
- `unverified`: blocked/network failure, unexpected content, missing reliable metadata, future/inconsistent dates. No assertion of latest version.

Even `metadata_found` **does not establish** an article was amended, applicability as of the case date, or that no other amendment exists. Review legal act / OJ / applicable dates manually for any regulatory recommendation.

In [7]:
OFFICIAL_HOSTS = {"eur-lex.europa.eu"}
MAX_RESPONSE_BYTES = 3_000_000
HTTP_TIMEOUT = (8, 15)

class NoExternalRedirects(requests.Session):
    def rebuild_auth(self, prepared_request, response):
        # Standard requests behavior; final hostname independently verified below.
        super().rebuild_auth(prepared_request, response)


def _iso_from_eu_date(value):
    try:
        return datetime.strptime(value, "%d/%m/%Y").date().isoformat()
    except ValueError:
        return None


def parse_eurlex_metadata(html, regulation, final_url):
    """Extract conservative basic-act-page consolidation metadata.

    Pure function: can unit-test offline. Does NOT state an act is in force.
    """
    basic = LAW_REGISTRY[regulation]
    expected_num = basic["celex"][1:]  # 2019R1793 for 32019R1793
    prefix = "0" + expected_num
    soup = BeautifulSoup(html, "html.parser")
    raw_text = soup.get_text(" ", strip=True)
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    matches = set()
    # Links on EUR-Lex basic act pages can include official consolidated CELEX references.
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        label = a.get_text(" ", strip=True)
        linked = urljoin(final_url, href)
        if urlparse(linked).hostname not in OFFICIAL_HOSTS:
            continue
        content = (href + " " + label).upper()
        for stamp in re.findall(re.escape(prefix)+r"-(\d{8})(?!\d)", content):
            try:
                d = datetime.strptime(stamp, "%Y%m%d").date()
                if d <= datetime.now(timezone.utc).date():
                    matches.add(d.isoformat())
            except ValueError:
                pass

    current_labels = re.findall(
        r"Access current version\s*\(\s*(\d{2}/\d{2}/\d{4})\s*\)",
        raw_text, flags=re.IGNORECASE
    )
    current_dates = [_iso_from_eu_date(x) for x in current_labels]
    current_dates = [x for x in current_dates if x and x <= datetime.now(timezone.utc).date().isoformat()]

    # Do not accept a random generic web page as the basic act:
    has_act_identity = basic["celex"].casefold() in (title+" "+raw_text[:2000]).casefold() or bool(matches) or bool(current_dates)
    candidates = sorted(matches | set(current_dates))
    return {
        "page_title": title[:300],
        "act_identity_detected": bool(has_act_identity),
        "consolidation_dates_in_official_links": sorted(matches),
        "current_version_dates_in_page_label": sorted(set(current_dates)),
        "latest_date_candidate": candidates[-1] if candidates else None,
        "version_metadata_observed": bool(candidates),
        "metadata_note": "Candidate dates from accessible EUR-Lex metadata, not legal in-force/applicability proof.",
    }


def retrieve_eurlex_basic_page(regulation, session=None):
    """Return a traceable evidence record; never raises for ordinary network failures."""
    if regulation not in LAW_REGISTRY:
        return {"regulation":regulation,"status":"unverified","error":"not_allowlisted"}
    law = LAW_REGISTRY[regulation]
    url = law["official_basic_url"]
    record = {
        "regulation":regulation, "basic_celex":law["celex"],
        "official_basic_url":url, "snapshot_date":law["snapshot_date"],
        "snapshot_file":law["snapshot_file"], "checked_utc":datetime.now(timezone.utc).isoformat(),
        "status":"unverified", "source":"EUR-Lex basic act page", "latest_date_candidate":None,
        "date_comparison":"not_verified", "article_level_changes_verified":False,
        "application_date_verified":False, "current_legal_status_verified":False,
    }
    try:
        client = session if session is not None else requests.Session()
        resp = client.get(url, headers={"User-Agent":"ScientificAIBuilder-RegulatoryResearch/0.1 (+noncommercial study)", "Accept":"text/html"}, timeout=HTTP_TIMEOUT)
        final_url = getattr(resp, "url", url)
        parsed_url = urlparse(final_url)
        record["resolved_url"] = final_url
        record["http_status"] = int(resp.status_code)
        if parsed_url.scheme != "https" or parsed_url.hostname not in OFFICIAL_HOSTS:
            record["error"] = "redirected_outside_eurlex"
            return record
        if resp.status_code != 200:
            record["error"] = f"http_{resp.status_code}"
            return record
        content = resp.content
        if len(content) > MAX_RESPONSE_BYTES:
            record["error"] = "html_response_too_large"
            return record
        if "html" not in resp.headers.get("Content-Type", "").lower():
            record["error"] = "not_html"
            return record
        metadata = parse_eurlex_metadata(resp.text, regulation, final_url)
        record.update(metadata)
        if not metadata["act_identity_detected"] or not metadata["version_metadata_observed"]:
            record["error"] = "no_verifiable_consolidation_metadata"
            return record
        record["status"] = "metadata_found"
        candidate = metadata["latest_date_candidate"]
        snapshot = law["snapshot_date"]
        if not snapshot:
            record["date_comparison"] = "original_snapshot_no_consolidation_date"
        elif candidate > snapshot:
            record["date_comparison"] = "newer_consolidation_metadata_found"
        elif candidate == snapshot:
            record["date_comparison"] = "same_date_in_observed_metadata_only"
        else:
            record["date_comparison"] = "observed_metadata_older_than_snapshot_review_needed"
        return record
    except requests.RequestException as ex:
        record["error"] = type(ex).__name__
        return record
    except Exception as ex:
        record["error"] = f"parse_error_{type(ex).__name__}"
        return record

## 6. Offline unit tests (no EUR-Lex/network/API)

Tests verify that only matching act metadata is accepted, a newer consolidation *date* is flagged, and blocked responses remain `unverified`. **Passing these tests does not prove live EUR-Lex availability.**

In [8]:
class _FakeResponse:
    def __init__(self, html, url, status=200, content_type="text/html"):
        self.text = html
        self.content = html.encode()
        self.url = url
        self.status_code = status
        self.headers = {"Content-Type":content_type}

class _FakeSession:
    def __init__(self, response): self.response = response
    def get(self, *args, **kwargs): return self.response

fake_url = LAW_REGISTRY["2019/1793"]["official_basic_url"]
valid_html = """<html><head><title>02019R1793 - EUR-Lex</title></head><body>
<a href="/legal-content/EN/TXT/?uri=CELEX%3A02019R1793-20260630">30/06/2026</a>
Access current version (30/06/2026)
</body></html>"""
record = retrieve_eurlex_basic_page("2019/1793", _FakeSession(_FakeResponse(valid_html, fake_url)))
assert record["status"] == "metadata_found" and record["latest_date_candidate"] == "2026-06-30"
assert record["date_comparison"] == "same_date_in_observed_metadata_only"
assert not record["current_legal_status_verified"]

blocked = retrieve_eurlex_basic_page("2019/1793", _FakeSession(_FakeResponse("<html>Forbidden</html>", fake_url, status=403)))
assert blocked["status"] == "unverified" and blocked["error"] == "http_403"

unknown = retrieve_eurlex_basic_page("2019/9999")
assert unknown["status"] == "unverified" and unknown["error"] == "not_allowlisted"

wrong_page = retrieve_eurlex_basic_page("2019/1793", _FakeSession(_FakeResponse("<html><title>Welcome</title><body>Hello</body></html>", fake_url)))
assert wrong_page["status"] == "unverified"
print("PASS: parser + allowlist + failure-closed behavior unit tests")

PASS: parser + allowlist + failure-closed behavior unit tests


## 7. Optional **live** EUR-Lex connectivity smoke test (no OpenAI API)

This makes one outbound HTTP request to official EUR-Lex. **Network failures are expected in some environments** and are not papered over with static claims. The agent tool uses the same function in Part 8.

Set `RUN_LIVE_EURLEX_PROBE=True` and rerun to test one law (2019/1793). May take up to ~25 seconds. Save evidence record separately.

In [10]:
RUN_LIVE_EURLEX_PROBE = True # set True for one official-site connectivity probe
LIVE_PROBE_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb06_eurlex_connectivity_probe.json")
if RUN_LIVE_EURLEX_PROBE:
    probe = retrieve_eurlex_basic_page("2019/1793")
    LIVE_PROBE_PATH.write_text(json.dumps(probe, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(probe, ensure_ascii=False, indent=2))
else:
    print("SKIPPED: live EUR-Lex connectivity probe (no network request).")

{
  "regulation": "2019/1793",
  "basic_celex": "32019R1793",
  "official_basic_url": "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A32019R1793",
  "snapshot_date": "2026-06-30",
  "snapshot_file": "EU_2019_1793_consolidated_2026-06-30.html",
  "checked_utc": "2026-10-19T11:04:32.048099+00:00",
  "status": "unverified",
  "source": "EUR-Lex basic act page",
  "latest_date_candidate": null,
  "date_comparison": "not_verified",
  "article_level_changes_verified": false,
  "application_date_verified": false,
  "current_legal_status_verified": false,
  "resolved_url": "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX%3A32019R1793",
  "http_status": 202,
  "error": "http_202"
}


In [11]:

import requests

url = (
    "https://eur-lex.europa.eu/legal-content/EN/TXT/"
    "?uri=CELEX:32019R1793"
)

try:
    response = requests.get(
        url,
        timeout=20,
        allow_redirects=True
    )

    print("HTTP status:", response.status_code)
    print("Final URL:", response.url)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Retry-After:", response.headers.get("Retry-After"))
    print("Response length:", len(response.content))
    print("Response preview:")
    print(response.text[:500])

except requests.RequestException as e:
    print("Connection failed:", str(e))


HTTP status: 202
Final URL: https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32019R1793
Content-Type: text/html; charset=UTF-8
Retry-After: None
Response length: 0
Response preview:



## 8. Register the **two** Build03 Function Tools

`search_regulation()` is the exact same local NB03-equivalent hybrid search as NB05 v2.

`check_current_law(regulation)` is **only** an official EUR-Lex metadata checker. It neither modifies the retriever nor pretends that an update in consolidation date proves substantive amendment or legal applicability. Every call is logged, including failure paths.

Both tools are independent. No hardcoded Q8 answers, no gold checklist injection.

In [12]:
RETRIEVAL_TOOL_LOG = []
CURRENT_LAW_TOOL_LOG = []

@function_tool
def search_regulation(query: str, top_k: int = 5) -> str:
    """Search the frozen five-law EU regulation library using NB03-equivalent hybrid retrieval.

    Args:
        query: Focused legal search query.
        top_k: Number of ranked chunks to return (1–10, default 5).
    """
    top_k = max(1, min(int(top_k), 10))
    results = hybrid_search(query, top_k=top_k)
    RETRIEVAL_TOOL_LOG.append({"timestamp_utc":datetime.now(timezone.utc).isoformat(),"query":query,"top_k":top_k,"results":results})
    return json.dumps([{
        "rank":r["rank"],"chunk_id":r["chunk_id"],"file":r["file"],
        "article":r["article"],"location":r["location"],"heading":r["heading"],"text":r["text"]
    } for r in results], ensure_ascii=False)

@function_tool
def check_current_law(regulation: str) -> str:
    """Check EUR-Lex official consolidation metadata for one of five known EU regulations.

    A newer consolidation date is only a date observation, NOT proof that an article changed,
    a legal provision currently applies, or no other amendments exist. If blocked, report unverified.

    Args:
        regulation: Regulation identifier, e.g. '2019/1793', '2023/915', '2017/625', '178/2002', or '2019/1013'.
    """
    regulation = regulation.strip()
    rec = retrieve_eurlex_basic_page(regulation)
    CURRENT_LAW_TOOL_LOG.append(rec)
    return json.dumps(rec, ensure_ascii=False)

print("Tools ready: search_regulation, check_current_law")

Tools ready: search_regulation, check_current_law


## 9. Controlled Build03 Q8 agent

Same shared context, same answer headings, same model and reasoning effort as frozen NB04b. Differences: tool-use instructions and dynamic tool budget are the **Build03 intervention**. Q8 gold checklist and manually verified reference remain outside generation.

**No assertion of current-law validity from metadata-only evidence.** A law may be amended with distinct effective dates, transitional clauses, corrigenda, etc. If official website is unavailable, final answer must say which checks could not be completed.

In [13]:
assert PARITY_OK
SHARED_CONTEXT = nb04b_config["benchmark_context"]
SHARED_SYSTEM_PROMPT = nb04b_config["system_prompt"]
BUILD03_TOOL_RULES = """You have two tools: search_regulation (frozen snapshot legal text)
and check_current_law (official EUR-Lex consolidation metadata for five known acts).
For material assertions, retrieve evidence. For Q8-style currentness analysis,
check official-source metadata when needed; report any unavailable/blocked checks as unverified.
State which topics require amendment, applicability-date, transitional-rule and annex-level checks.
A newer consolidated date does not prove changed provisions, legal effectiveness, or current applicability.
Do not infer absence of amendments from a failed or incomplete lookup.
Never assert an actual current limit, tariff code, check rate, or border action without specific verified evidence.
Use regulation numbers and official source links when provided by tools.
Be conservative; distinguish confirmed source observations from unverified legal questions.
""".strip()
AGENT_INSTRUCTIONS = "\n\n".join([SHARED_CONTEXT,SHARED_SYSTEM_PROMPT,BUILD03_TOOL_RULES])

regulatory_agent = Agent(
    name="Build03 Regulatory Research Agent NB06",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning={"effort":REASONING_EFFORT},
        max_tokens=int(nb04b_config["max_output_tokens"]),
    ),
    instructions=AGENT_INSTRUCTIONS,
    tools=[search_regulation, check_current_law],
)
print("Agent ready | model:", MODEL_NAME, "| reasoning:", REASONING_EFFORT)
print("Q8:", Q8)

Agent ready | model: gpt-6-luna | reasoning: medium
Q8: Which parts of the assessment depend on current legislation and therefore require checking for amendments or updates?


### 9.1 EUR-LEX connection 

In [14]:

import requests
import xml.etree.ElementTree as ET

def test_cellar_connection(celex="32019R1793"):
    url = (
        f"https://publications.europa.eu/"
        f"resource/celex/{celex}"
    )

    headers = {
        "Accept": "application/rdf+xml;notice=non-inferred"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30,
            allow_redirects=True
        )

        print("HTTP status:", response.status_code)
        print("Final URL:", response.url)
        print("Content-Type:", response.headers.get("Content-Type"))
        print("Response size:", len(response.content))

        if response.status_code != 200 or not response.content:
            print("UNVERIFIED: No usable response.")
            return None

        # Verify that the response is valid XML/RDF
        root = ET.fromstring(response.content)

        print("PASS: Valid XML/RDF response received.")
        print("Root tag:", root.tag)
        print("Preview:", response.text[:300])

        return root

    except requests.RequestException as exc:
        print("NETWORK ERROR:", exc)
        return None

    except ET.ParseError as exc:
        print("INVALID XML:", exc)
        return None


cellar_result = test_cellar_connection()


HTTP status: 200
Final URL: http://publications.europa.eu/resource/cellar/06fd8149-fa2d-11e9-8c1f-01aa75ed71a1/rdf/object/noninferred
Content-Type: application/rdf+xml;charset=UTF-8
Response size: 46497
PASS: Valid XML/RDF response received.
Root tag: {http://www.w3.org/1999/02/22-rdf-syntax-ns#}RDF
Preview: <rdf:RDF
    xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    xmlns:j.0="http://publications.europa.eu/ontology/cdm#"
    xmlns:owl="http://www.w3.org/2002/07/owl#"
    xmlns:j.1="http://www.w3.org/2004/02/skos/core#"
    xmlns:j.2="http://publications.europa.eu/ontology/annotation#"
    


In [15]:

import re
from datetime import datetime, timezone
from xml.etree import ElementTree as ET

RDF_NS = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}"

def inspect_cellar_consolidations(
    root,
    celex="32019R1793",
    snapshot_date="2026-06-30"
):
    original_uri = (
        "http://publications.europa.eu/resource/celex/"
        + celex
    )

    links = []

    for element in root.iter():
        subject = element.attrib.get(RDF_NS + "about", "")

        if subject.rstrip("/") != original_uri:
            continue

        for child in element:
            predicate = child.tag.split("}")[-1]
            target = child.attrib.get(RDF_NS + "resource", "")

            if "consolidat" in predicate.lower():
                links.append({
                    "predicate": predicate,
                    "target": target
                })

    expected_prefix = "0" + celex[1:]

    dates = set()
    for link in links:
        target = link["target"]
        match = re.search(
            re.escape(expected_prefix) + r"-(\d{8})",
            target
        )
        if match:
            try:
                date_str = datetime.strptime(
                    match.group(1), "%Y%m%d"
                ).date().isoformat()
                dates.add(date_str)
            except ValueError:
                pass

    dates = sorted(dates)
    latest = dates[-1] if dates else None

    print("Original CELEX:", celex)
    print("Snapshot date:", snapshot_date)
    print("Consolidation links found:", len(links))
    print("Consolidation dates:", dates)
    print("Latest date candidate:", latest)

    if not latest:
        status = "unverified"
        print("STATUS: UNVERIFIED — no consolidation date found")
    elif latest > snapshot_date:
        status = "newer_consolidation_found"
        print("STATUS: NEWER CONSOLIDATION FOUND")
    elif latest == snapshot_date:
        status = "snapshot_matches_latest_candidate"
        print("STATUS: SNAPSHOT DATE MATCHES LATEST CANDIDATE")
    else:
        status = "snapshot_date_newer_than_metadata"
        print("STATUS: REVIEW REQUIRED — unexpected date comparison")

    return {
        "celex": celex,
        "snapshot_date": snapshot_date,
        "consolidation_dates": dates,
        "latest_date_candidate": latest,
        "status": status,
        "source": "EU Publications Office Cellar",
        "article_level_changes_verified": False,
        "current_legal_status_verified": False,
        "checked_utc": datetime.now(timezone.utc).isoformat()
    }


cellar_metadata = inspect_cellar_consolidations(
    cellar_result,
    celex="32019R1793",
    snapshot_date="2026-06-30"
)


Original CELEX: 32019R1793
Snapshot date: 2026-06-30
Consolidation links found: 0
Consolidation dates: []
Latest date candidate: None
STATUS: UNVERIFIED — no consolidation date found


In [16]:

from collections import Counter

RDF_RESOURCE = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource"
RDF_ABOUT = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about"

def local_name(tag):
    return tag.split("}")[-1]

# 1. Find all consolidation-related properties anywhere in the RDF
matches = []

for parent in cellar_result.iter():
    for child in parent:
        predicate = local_name(child.tag)
        target = child.attrib.get(RDF_RESOURCE, "")
        value = (child.text or "").strip()

        if any(word in (predicate + target + value).lower()
               for word in ("consolidat", "02019r1793")):
            matches.append({
                "subject": parent.attrib.get(RDF_ABOUT, ""),
                "predicate": predicate,
                "target": target,
                "value": value[:150]
            })

print("Consolidation-related RDF properties:", len(matches))

for item in matches[:30]:
    print("\nSubject:", item["subject"])
    print("Predicate:", item["predicate"])
    print("Target:", item["target"])
    print("Value:", item["value"])

# 2. Inspect which predicates occur in the RDF
counts = Counter(
    local_name(child.tag)
    for parent in cellar_result.iter()
    for child in parent
)

print("\nMost common RDF properties:")
for name, count in counts.most_common(25):
    print(f"{name}: {count}")


Consolidation-related RDF properties: 0

Most common RDF properties:
Description: 57
type: 57
annotatedTarget: 51
annotatedProperty: 51
annotatedSource: 51
type_of_link_target: 39
start_of_validity: 38
resource_legal_implicitly_repeals_resource_legal: 34
work_is_about_concept_eurovoc: 10
work_cites_work: 10
article: 6
comment_on_legal_basis: 6
paragraph: 5
comment_on_date: 5
resource_legal_is_about_subject-matter: 5
resource_legal_repeals_resource_legal: 5
inScheme: 4
work_id_document: 4
sameAs: 3
type_of_date: 2
resource_legal_date_deadline: 2
resource_legal_is_about_concept_directory-code: 2
resource_legal_date_entry-into-force: 2
work_created_by_agent: 2
resource_legal_based_on_resource_legal: 2


In [17]:

import requests
import re
from datetime import datetime

SPARQL_ENDPOINT = (
    "https://publications.europa.eu/webapi/rdf/sparql"
)

def find_consolidated_versions(celex="32019R1793"):

    query = f"""
    PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>

    SELECT DISTINCT ?consolidated ?consolidated_celex
    WHERE {{
        ?original cdm:work_id_document "{celex}" .

        ?consolidated
            cdm:act_consolidated_consolidates_resource_legal
            ?original .

        ?consolidated
            cdm:work_id_document
            ?consolidated_celex .
    }}
    ORDER BY DESC(?consolidated_celex)
    """

    try:
        response = requests.get(
            SPARQL_ENDPOINT,
            params={"query": query},
            headers={
                "Accept": "application/sparql-results+json"
            },
            timeout=45
        )

        print("HTTP status:", response.status_code)
        response.raise_for_status()

        data = response.json()
        bindings = data["results"]["bindings"]

        versions = []

        for item in bindings:
            version_celex = item[
                "consolidated_celex"
            ]["value"]

            # Standard consolidated CELEX format:
            # 02019R1793-YYYYMMDD
            match = re.fullmatch(
                r"02019R1793-(\d{8})",
                version_celex
            )

            if match:
                version_date = datetime.strptime(
                    match.group(1), "%Y%m%d"
                ).date().isoformat()

                versions.append({
                    "celex": version_celex,
                    "date": version_date,
                    "uri": item["consolidated"]["value"]
                })

        versions.sort(
            key=lambda x: x["date"],
            reverse=True
        )

        print("Consolidated versions found:", len(versions))

        for version in versions[:10]:
            print(version["date"], version["celex"])

        return versions

    except (requests.RequestException, ValueError, KeyError) as e:
        print("UNVERIFIED:", str(e))
        return None


versions = find_consolidated_versions("32019R1793")

if versions:
    latest = versions[0]

    print("\nLATEST CONSOLIDATION CANDIDATE")
    print("CELEX:", latest["celex"])
    print("Date:", latest["date"])
    print("Snapshot date:", "2026-06-30")

elif versions == []:
    print(
        "\nNo matching consolidated versions returned. "
        "Currentness remains UNVERIFIED."
    )


HTTP status: 200
Consolidated versions found: 0

No matching consolidated versions returned. Currentness remains UNVERIFIED.


## 10. Official Q8 test — **paid API call + live EUR-Lex requests**

**This cell incurs API costs and may make multiple EUR-Lex network requests.** Jupyter runs SDK Agents with `await Runner.run`, not `run_sync`.

Guard against accidental overwriting: a completed run file will block rerun unless you deliberately archive/rename it. Network/tool logs are retained for independent audit.

Before running, ensure you can reach EUR-Lex; if not, a run can still test *honest uncertainty handling*, but cannot demonstrate successful real-time verification.

In [ ]:
Q8_RUN_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb06_q8_official.json")
assert PARITY_OK
if Q8_RUN_PATH.exists():
    raise FileExistsError(f"A Q8 official run already exists: {Q8_RUN_PATH}. Archive it before deliberate rerun.")
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
assert os.getenv("OPENAI_API_KEY"), "API key required for Q8 run"

RETRIEVAL_TOOL_LOG.clear()
CURRENT_LAW_TOOL_LOG.clear()

q8_result = await Runner.run(regulatory_agent, Q8, max_turns=18)
q8_answer = str(q8_result.final_output)

# Save the complete tool records promptly, in the same execution cell.
q8_record = {
    "build":"Build03", "experiment":"q8_current_law_checker_official_nb06",
    "status":"run_completed_evaluation_pending",
    "benchmark_id":benchmark["benchmark_id"],"benchmark_sha256":benchmark_hash,
    "question_id":"Q8","question":Q8,"model":MODEL_NAME,
    "reasoning_effort":REASONING_EFFORT,
    "max_output_tokens":int(nb04b_config["max_output_tokens"]),
    "shared_benchmark_context":SHARED_CONTEXT,
    "shared_system_prompt":SHARED_SYSTEM_PROMPT,
    "build03_tool_rules":BUILD03_TOOL_RULES,
    "baseline_retrieval_config":retrieval_config,
    "parity":{"passed":PARITY_OK,"matched_ordered_chunks":matched,"total_ordered_chunks":50,"audit_file":str(PARITY_PATH)},
    "local_search_call_count":len(RETRIEVAL_TOOL_LOG),
    "current_law_call_count":len(CURRENT_LAW_TOOL_LOG),
    "local_search_tool_calls":RETRIEVAL_TOOL_LOG,
    "current_law_tool_calls":CURRENT_LAW_TOOL_LOG,
    "final_output":q8_answer,
    "run_at_utc":datetime.now(timezone.utc).isoformat(),
}
with Q8_RUN_PATH.open("x", encoding="utf-8") as f:
    json.dump(q8_record, f, ensure_ascii=False, indent=2)
print("Saved official Q8 run:", Q8_RUN_PATH)
print("\n===== BUILD03 Q8 FINAL ANSWER =====\n", q8_answer)
print("\nTool calls: local retrieval =", len(RETRIEVAL_TOOL_LOG), "; current law =", len(CURRENT_LAW_TOOL_LOG))
for c in CURRENT_LAW_TOOL_LOG:
    print(c["regulation"], "|", c["status"], "|",c.get("latest_date_candidate"),"|",c.get("date_comparison"),"|",c.get("error", ""))

## 11. Frozen Build02 Q8 vs new Build03 Q8 (no baseline regeneration)

The previous NB04b Build02 result and official Notion grades stay fixed. Do **not** automatically claim `Yes` for correctness/grounding/completeness because a website returned a date. Review tool traces and the legal provisions separately.

In [ ]:
print("===== FROZEN NB04b BUILD02 Q8 RAG =====\n")
print(FROZEN_Q8_ANSWER)
if Q8_RUN_PATH.exists():
    official = json.loads(Q8_RUN_PATH.read_text(encoding="utf-8"))
    print("\n===== BUILD03 NB06 Q8 =====\n")
    print(official["final_output"])
    print("\nOfficial-source outcomes:")
    for r in official["current_law_tool_calls"]:
        print(r["regulation"],r["status"],r.get("date_comparison"),r.get("official_basic_url"))
else:
    print("Q8 paid run not yet executed; frozen baseline still available.")

## 12. Frozen Q8 evaluation worksheet (human-reviewed, not model-generated)

**Q8 must-have checklist (Notion, 2026-10-08 frozen reference)**
1. Current 2023/915 limits, product scope and Art. 3 implications.
2. Current 2019/1793 Annex II, hazards, CN/TARIC codes and check frequency.
3. Entry documents/procedures (2019/1793 Arts 9–11 / Annex IV; 2017/625; 2019/1013).
4. Enforcement/current legal versions.
5. Assessment date, applicability dates, amendments and transitional provisions.
6. Static RAG is only a legal snapshot.

**Scoring:** `Correct` / `Grounded` / `Complete` each in `Yes`, `Partial`, `No`. Currentness verification *cannot* be scored fully grounded if all official lookups return `unverified`. A checked consolidated date is evidence of publication/version metadata, not article-level legal correctness.

This checklist is presented **after** generation, not provided to the agent.

In [ ]:
EVALUATION_PATH = __import__("pathlib").Path("experiments/build03_diagnostics/data/build03_nb06_q8_evaluation_worksheet.csv")
if EVALUATION_PATH.exists():
    print("Existing evaluation worksheet preserved:", EVALUATION_PATH)
else:
    worksheet = pd.DataFrame([
        {"qid":"Q8", "build":"Build02_frozen", "correct":"Yes", "grounded":"Yes", "complete":"Partial", "note":"Notion frozen Q8 evaluation"},
        {"qid":"Q8", "build":"Build03_NB06", "correct":"", "grounded":"", "complete":"", "note":"HUMAN REVIEW: check original evidence + actual EUR-Lex tool outcomes"},
    ])
    worksheet.to_csv(EVALUATION_PATH,index=False,encoding="utf-8-sig")
    print("Created:", EVALUATION_PATH)
    display(worksheet)

## 13. Final audit checklist and interpretation

- [ ] Same benchmark SHA-256, **1383** chunks and five frozen regulations.
- [ ] **50/50** ordered NB03 retrieval parity; STOP if not.
- [ ] Two SDK tools registered with `@function_tool`, no obsolete `@tool`.
- [ ] Offline parser tests pass and network availability separately recorded.
- [ ] `check_current_law` only returns **actual retrieved official metadata** (or explicit `unverified`).
- [ ] No invented current legal rates, limits or amendment claims.
- [ ] Q8 uses same NB04b benchmark context, model, reasoning effort and output format.
- [ ] Official Q8 result saved **separately**; old Q7 pilot/v2 and frozen NB04b never overwritten.
- [ ] Human review of Q8 checklist and verified source links before updating project conclusions.

**Research note:** With a second tool, Build03 differs from Build02 in both *retrieval policy* and *available official-source evidence*. Treat any improvement as tool-enabled evidence acquisition, not evidence of autonomous planning gains. Exact provision lookup remains an optional independent diagnostic for the Q7 local-library recall failure.

**What this v1 intentionally does not claim:** current legal validity, complete amendment history, verified effective dates, annex-row diff, exact jurisdictional enforcement applicability. Those require an additional structured amendment/current-law verification pipeline.